실제 HDI_Relay에 넣을최종 제안본

In [1]:
import torch
import numpy as np
import pandas as pd
import json

# ==========================================
# 🧠 [모듈 1] Core ML Engine (범용)
# ==========================================
def detect_anomaly(model, scaler, tick_df, feature_cols):
    """들어온 1틱의 데이터에 대해 건강도(오차 점수)와 변수별 오차량을 계산합니다."""
    model.eval() 
    with torch.no_grad():
        scaled_data = scaler.transform(tick_df[feature_cols])
        tensor_data = torch.FloatTensor(scaled_data)
        
        reconstructed = model(tensor_data)
        mse_loss = torch.mean((tensor_data - reconstructed) ** 2).item()
        feature_errors = torch.abs(tensor_data - reconstructed).numpy()[0] 
        
    return mse_loss, feature_errors

# ==========================================
# 📦 함수 2: 문맥 데이터 추출 (Context Window)
# ==========================================
def extract_context_data(buffer_df, current_idx, lookback=5):
    """이상이 발생했을 때, 룰 기반 판단을 위해 '해당 틱 포함 이전 N개의 틱'을 확보합니다."""
    start_idx = max(0, current_idx - lookback + 1)
    return buffer_df.iloc[start_idx : current_idx + 1].copy()

# ==========================================
# 🕵️‍♂️ 함수 3: 범인 색출 (Feature Attribution)
# ==========================================
def identify_culprit(feature_errors, feature_cols, top_k=3):
    """변수별 오차량을 바탕으로 이번 이상 수치에 가장 크게 기여한 범인을 뽑아냅니다."""
    total_error = np.sum(feature_errors)
    if total_error == 0:
        return pd.DataFrame()
    
    contributions = (feature_errors / total_error) * 100
    
    culprit_df = pd.DataFrame({
        'Feature': feature_cols,
        'Error_Contribution(%)': contributions
    }).sort_values(by='Error_Contribution(%)', ascending=False)
    
    return culprit_df.head(top_k)

# ==========================================
# 
# [모듈 2] Business Logic Engine (도메인 맞춤형)
# ==========================================
def generate_fact_bomb(culprits_df, context_df, meta_info, pump_id, config=None):
    """
    AI가 색출한 범인(Feature)과 현재 상황을 바탕으로, 
    해당 펌프(pump_id)에 맞는 현장 맞춤형 조치 사항을 뱉어냅니다.
    """
    # 1. Config 초기화 (입력이 없으면 하드웨어 스펙 기반 기본값 사용)
    if config is None:
        config = {
            'MAX_SV_SCALE': 16000.0,
            'Ana_JUMP_LIMIT': 1000.0,
            'Ana_DROP_LIMIT': 1000.0,
            'ANA_OUT_HIGH_LIMIT': 14000.0,
            'PT_MAX_LIMIT': 220.0,
            'PT_NORMAL_MIN': 80.0
        }

    # 2. 메타 정보 추출
    wagon_num = meta_info.get('Wagon_Num', '알수없음')
    tick_idx = meta_info.get('Tick_Index', 0)
    
    # 3. 범인 지분율 변환 (DataFrame -> Dict) 및 Top 1 추출
    if culprits_df.empty:
        return "데이터 오류: culprits 정보가 없습니다."
    
    top1_feature = culprits_df.iloc[0]['Feature']
    attributions = dict(zip(culprits_df['Feature'], culprits_df['Error_Contribution(%)'] / 100.0))
    
    # 4. 펌프 ID 동적 태그명 생성
    tag_SV = f'g_s_SV_{pump_id}'
    tag_Ana = f'Ana_Out_{pump_id}'
    tag_PT = f'Scale_Out___PT_{pump_id}'
    tag_FT = f'Scale_Out___FT_{pump_id}'
    tag_Temp = f'TK_Temp_PV_{pump_id}' # 온도는 앞자리 매핑용

    # 5. 현재 틱 데이터 및 Ana_Out_Diff 계산
    current_row = context_df.iloc[-1].to_dict()
    if len(context_df) >= 2:
        ana_out_diff = context_df.iloc[-1].get(tag_Ana, 0) - context_df.iloc[-2].get(tag_Ana, 0)
    else:
        ana_out_diff = 0.0

    # 6. 공통 Context 변수 및 지분율 추출
    is_start = (current_row.get('Phase_Start', 0) == 1.0)
    is_steady = (current_row.get('Phase_Steady', 0) == 1.0)
    is_transition = (current_row.get('Phase_Transition', 0) == 1.0)
    
    pt_val = current_row.get(tag_PT, 0)
    pt_diff = current_row.get('Rolling_PT_Diff_3', 0)
    ft_error_rate = current_row.get('Instant_FT_Error_Rate', 0)
    ana_out = current_row.get(tag_Ana, 0)
    
    w_pt = attributions.get(tag_PT, 0) + attributions.get('Rolling_PT_Max_3', 0)
    w_ana = attributions.get(tag_Ana, 0)
    w_sv = attributions.get(tag_SV, 0) + attributions.get('Prev_SV', 0)
    w_ft_total = attributions.get('Instant_FT_Error_Rate', 0) + attributions.get(tag_FT, 0)

    # --------------------------------------------------
    # 🚀 리포트 헤더 생성
    # --------------------------------------------------
    report = f" [펌프: {pump_id} | 대차: {wagon_num} / {tick_idx}틱] 긴급 진단 리포트\n"

    # =========================================================
    # ⚡ Rule 1. 인버터 트립 시나리오 (OC, OU3, OLU)
    # =========================================================
    ctrl_features = [tag_Ana, tag_SV, 'Prev_SV_Diff', 'Instant_SV_Diff']
    if top1_feature in ctrl_features or 'Ana' in top1_feature:
        
        if ana_out_diff > config['Ana_JUMP_LIMIT']:
            if is_start or is_transition:
                if w_ana > 0.10 or top1_feature == tag_Ana:
                    report += f"[급가속 과전류(OC) 트립 위험]\n"
                    report += f"[현상] 인버터 출력(Ana_Out)이 +{ana_out_diff:.0f} 스케일 급증하여 모터에 엄청난 가속 충격이 가해졌습니다.\n"
                    report += f"[즉각 조치] 인버터 '가속 시간(Acc Time)'을 늘리거나 레시피 단차를 줄이세요."
                    return report
                    
        elif ana_out_diff < -config['Ana_DROP_LIMIT']:
            if w_ana > 0.10 or top1_feature == tag_Ana:
                report += f" [급감속 고전압(OU3) 트립 위험]\n"
                report += f" [현상] 인버터 출력({ana_out_diff:.0f} 스케일) 급감으로 모터 회생 전압이 발생했습니다.\n"
                report += f" [즉각 조치] '감속 시간(Dec Time)'을 늘려주세요. 반복 시 제동 저항 또는 인버터 커패시터를 점검하세요."
                return report
                
        elif top1_feature in [tag_Ana, tag_SV]:
            if ana_out > config['ANA_OUT_HIGH_LIMIT'] and tick_idx >= 3:
                current_mA = 4.0 + (ana_out / 16000.0) * 16.0
                report += f" [인버터 과부하 (OLU/EOCR)]\n"
                report += f" [현상] 지령치(Ana_Out)가 {ana_out:.0f} 스케일(약 {current_mA:.1f}mA)로 장기 유지되어 모터 용량을 초과했습니다.\n"
                report += f" [즉각 조치] EOCR 리셋 후 레시피 타겟(SV) 하향을 권장합니다."
                return report

        if top1_feature == tag_Ana and pt_val < 5.0 and ana_out > config['ANA_OUT_HIGH_LIMIT']:
            report += f" [치명적 결함: 샤프트 파단]\n"
            report += f" [현상] 제어 신호는 풀악셀인데 실제 압력이 {pt_val} bar입니다.\n"
            report += f" [즉각 조치] 모터-펌프 연결 샤프트 파단 또는 배관 완전 파열입니다. 즉시 라인을 정지하세요!"
            return report

    # =========================================================
    # 🌊 Rule 2. 토출 미달 및 승압 실패 (유량/압력 절대치)
    # =========================================================
    if top1_feature in [tag_PT, 'Rolling_PT_Max_3', 'Instant_FT_Error_Rate', tag_FT , 'Rolling_PT_Diff_3']:
        
        if (top1_feature in ['Instant_FT_Error_Rate', tag_FT] or w_ft_total > 0.10) and ft_error_rate > 15.0:
            if is_start:
                if w_pt > 0.10:
                    report += f" [초기 승압 실패]\n"
                    report += f" [현상] 압력이 형성되지 않아 심각한 토출 미달(오차율: {ft_error_rate:.1f}%)이 발생했습니다.\n"
                    report += f" [즉각 조치] 올림프스 세트 마모 또는 펌프 토출 불량을 점검하세요."
                    return report
                else:
                    report += f" [초기 토출 미달]\n"
                    report += f" [현상] 압력은 정상이나 유량이 목표치를 크게 밑돕니다(오차율: {ft_error_rate:.1f}%).\n"
                    report += f" [즉각 조치] 배관 내 초기 에어(Air) 혼입 또는 원료 공급 라인을 점검하세요."
                    return report
            else:
                report += f" [운전 중 토출 급감]\n"
                report += f" [현상] 운전 중 실제 유량(FT)이 목표(SV) 대비 비정상적으로 꺼졌습니다(오차율: {ft_error_rate:.1f}%).\n"
                report += f" [즉각 조치] 펌프 순간 슬립 또는 원료 탱크 캐비테이션을 의심하세요."
                return report

        elif pt_val > config['PT_MAX_LIMIT'] or current_row.get('Rolling_PT_Max_3', 0) > config['PT_MAX_LIMIT']:
            if w_pt > 0.30 or top1_feature in [tag_PT, 'Rolling_PT_Max_3']:
                report += f" [절대 압력 높음]\n"
                report += f" [현상] 현재 압력({pt_val} bar)이 시스템 최고 한계치를 초과했습니다.\n"
                report += f" [즉각 조치] 설비 보호를 위해 즉시 조압 밸브를 열어 압력을 낮추세요."
                return report
        
        elif top1_feature in [tag_PT, 'Rolling_PT_Max_3','Rolling_PT_Diff_3']:
            report += f" [상대적 압력 높음 / 조압 불량]\n"
            report += f" [현상]  평소 설정된 기준치 대비 압력이 비정상적으로 높게 형성되고 있습니다 (현재: {pt_val} bar).\n"
            report += f" [즉각 조치]목표하는 적정 압력으로 낮춰주세요."
            return report

        elif top1_feature == 'Instant_FT_Error_Rate' and w_pt < 0.05 and w_ana < 0.05:
            report += f" [계측기/노이즈]\n"
            report += f" [현상] 제어와 압력은 평온하나 유량 오차가 감지되었습니다.\n"
            report += f" [즉각 조치] 배관 내 잔여 에어 통과 또는 플로우메터 센서 노이즈입니다."
            return report

    # =========================================================
    # ⚙️ Rule 3. 펌프 마모 및 배관 파열 (누적/변화량)
    # =========================================================
    if top1_feature == 'Cum_FT_Error':
        if is_steady:
            if pt_diff < 0 or pt_val < config['PT_NORMAL_MIN']:
                report += f" [펌프/카플링 마모]\n"
                report += f" [현상] 압력({pt_val} bar) 저하와 함께 유량 오차가 걷잡을 수 없이 누적됩니다.\n"
                report += f" [즉각 조치] 펌프 슬립 현상 확정. 펌프 및 카플링 교체 계획을 수립하세요."
                return report
            else:
                report += f" [유량 누적 미달]\n"
                report += f" [현상] 압력은 유지되나 유량 오차가 지속적으로 누적됩니다.\n"
                report += f" [즉각 조치] 노즐 끝단 미세 막힘 또는 원료 굳음을 점검하세요."
                return report

    elif top1_feature == 'Rolling_PT_Diff_3':
        if is_steady and pt_diff < -20.0:
            report += f" [유압 붕괴]\n"
            report += f" [현상] 안정 구간에서 압력이 순식간에 급락({pt_diff} bar)했습니다!\n"
            report += f" [즉각 조치] 배관 파열(누유) 또는 펌프 내부 기어 파손을 즉시 점검하세요."
            return report

    # =========================================================
    # 🌡️ Rule 4. 환경 주도형
    # =========================================================
    if top1_feature.startswith(tag_Temp[:-2]): # 'TK_Temp_PV_' 로 시작하는지 유연하게 확인
        report += f" [원료 점도 이상]\n"
        report += f" [현상] 제어는 정상하나 원료 온도가 비정상적 기여도를 보입니다.\n"
        report += f" [즉각 조치] 폼(Foam) 품질 불량 위험이 큽니다. 칠러/히터 및 원료 점도를 점검하세요."
        return report

    # =========================================================
    # 🔍 Catch-All 방어 로직
    # =========================================================
    top_score = culprits_df.iloc[0]['Error_Contribution(%)']
    report += f" [복합 이상 패턴 감지]\n"
    report += f" [현상] 지정된 기계/전기적 알람 조건을 벗어난 비정상 데이터 구조 탐지 (주 원인: {top1_feature}, 기여도 {top_score:.1f}%).\n"
    report += f" [즉각 조치] 대시보드 시계열 그래프를 띄워 작업자와 엔지니어가 복합적인 원인을 함께 점검하세요."
    
    return report

In [2]:
def add_report_to_json_payload(json_payload):
    """
    입력 JSON(payload)에 report 키를 추가합니다.
    - str(JSON 문자열) / dict 1건 / list[dict] 모두 지원
    - 각 item에 'report': <진단문자열> 추가
    """
    import json

    def _attach_report(item):
        pump_id = item.get('pump_id', 'Unknown')
        wagon_num = item.get('wagon_num', 'Unknown')
        tick_idx = item.get('tick_idx', 0)

        meta_info = {
            'Wagon_Num': wagon_num,
            'Tick_Index': tick_idx
        }

        culprits_df = pd.DataFrame(item.get('culprits_analysis', []))
        context_df = pd.DataFrame(item.get('context_history', []))

        report_msg = generate_fact_bomb(culprits_df, context_df, meta_info, pump_id)

        updated = dict(item)
        updated['report'] = report_msg
        return updated

    # JSON 문자열이 들어오면 먼저 파싱
    if isinstance(json_payload, str):
        json_payload = json.loads(json_payload)

    if isinstance(json_payload, dict):
        return _attach_report(json_payload)

    if isinstance(json_payload, list):
        return [_attach_report(item) for item in json_payload]

    raise TypeError('json_payload must be str, dict, or list of dicts')

# 사용 예시
# final_output = add_report_to_json_payload(test_json_data)
# print(json.dumps(final_output, indent=4, ensure_ascii=False))
# print(final_output['report'])

In [9]:
test_json_data = """
{"timestamp": "2026-04-16T10:29:58", "pump_id": "I7", "wagon_num": 29.0, "tick_idx": 3, "mse_score": 1.0386, "culprits_analysis": [{"Feature": "Cum_FT_Error", "Error_Contribution(%)": 35.1}, {"Feature": "Phase_Transition", "Error_Contribution(%)": 18.0}, {"Feature": "Rolling_PT_Diff_3", "Error_Contribution(%)": 8.1}], "context_history": [{"g_s_SV_I7": 1200.0, "Prev_SV": 1200.0, "Prev_SV_Diff": 0.0, "Ana_Out_I7": 0.0, "TK_Temp_PV_I7": 0.0, "Scale_Out___PT_I7": 63.0, "Rolling_PT_Max_3": 0.0, "Rolling_PT_Diff_3": 0.0, "Scale_Out___FT_I7": 1218.0, "Instant_FT_Error_Rate": -1.5, "Cum_FT_Error": 0.0, "Phase_Start": 0.0, "Phase_Steady": 1.0, "Phase_Transition": 0.0, "Wagon_Num": 29.0, "Tick_Index": 3}, {"g_s_SV_I7": 1030.0, "Prev_SV": 1200.0, "Prev_SV_Diff": -170.0, "Ana_Out_I7": 0.0, "TK_Temp_PV_I7": 0.0, "Scale_Out___PT_I7": 62.0, "Rolling_PT_Max_3": 0.0, "Rolling_PT_Diff_3": 0.0, "Scale_Out___FT_I7": 1355.0, "Instant_FT_Error_Rate": -31.553398, "Cum_FT_Error": 0.0, "Phase_Start": 1.0, "Phase_Steady": 0.0, "Phase_Transition": 1.0, "Wagon_Num": 29.0, "Tick_Index": 0}, {"g_s_SV_I7": 1030.0, "Prev_SV": 1030.0, "Prev_SV_Diff": 0.0, "Ana_Out_I7": 0.0, "TK_Temp_PV_I7": 0.0, "Scale_Out___PT_I7": 52.0, "Rolling_PT_Max_3": 0.0, "Rolling_PT_Diff_3": 0.0, "Scale_Out___FT_I7": 1343.0, "Instant_FT_Error_Rate": -30.38835, "Cum_FT_Error": 0.0, "Phase_Start": 0.0, "Phase_Steady": 1.0, "Phase_Transition": 0.0, "Wagon_Num": 29.0, "Tick_Index": 1}, {"g_s_SV_I7": 1030.0, "Prev_SV": 1030.0, "Prev_SV_Diff": 0.0, "Ana_Out_I7": 0.0, "TK_Temp_PV_I7": 0.0, "Scale_Out___PT_I7": 72.0, "Rolling_PT_Max_3": 0.0, "Rolling_PT_Diff_3": 0.0, "Scale_Out___FT_I7": 1057.0, "Instant_FT_Error_Rate": -2.621359, "Cum_FT_Error": 0.0, "Phase_Start": 0.0, "Phase_Steady": 1.0, "Phase_Transition": 0.0, "Wagon_Num": 29.0, "Tick_Index": 2}, {"g_s_SV_I7": 1360.0, "Prev_SV": 1030.0, "Prev_SV_Diff": 330.0, "Ana_Out_I7": 0.0, "TK_Temp_PV_I7": 0.0, "Scale_Out___PT_I7": 88.0, "Rolling_PT_Max_3": 0.0, "Rolling_PT_Diff_3": 0.0, "Scale_Out___FT_I7": 1405.0, "Instant_FT_Error_Rate": -3.308824, "Cum_FT_Error": 0.0, "Phase_Start": 0.0, "Phase_Steady": 0.0, "Phase_Transition": 1.0, "Wagon_Num": 29.0, "Tick_Index": 3}]}

"""

result = add_report_to_json_payload(test_json_data)
print(json.dumps(result, ensure_ascii=False, indent=2))
print("\n[report]")
print(result['report'])

{
  "timestamp": "2026-04-16T10:29:58",
  "pump_id": "I7",
  "wagon_num": 29.0,
  "tick_idx": 3,
  "mse_score": 1.0386,
  "culprits_analysis": [
    {
      "Feature": "Cum_FT_Error",
      "Error_Contribution(%)": 35.1
    },
    {
      "Feature": "Phase_Transition",
      "Error_Contribution(%)": 18.0
    },
    {
      "Feature": "Rolling_PT_Diff_3",
      "Error_Contribution(%)": 8.1
    }
  ],
  "context_history": [
    {
      "g_s_SV_I7": 1200.0,
      "Prev_SV": 1200.0,
      "Prev_SV_Diff": 0.0,
      "Ana_Out_I7": 0.0,
      "TK_Temp_PV_I7": 0.0,
      "Scale_Out___PT_I7": 63.0,
      "Rolling_PT_Max_3": 0.0,
      "Rolling_PT_Diff_3": 0.0,
      "Scale_Out___FT_I7": 1218.0,
      "Instant_FT_Error_Rate": -1.5,
      "Cum_FT_Error": 0.0,
      "Phase_Start": 0.0,
      "Phase_Steady": 1.0,
      "Phase_Transition": 0.0,
      "Wagon_Num": 29.0,
      "Tick_Index": 3
    },
    {
      "g_s_SV_I7": 1030.0,
      "Prev_SV": 1200.0,
      "Prev_SV_Diff": -170.0,
      "Ana_Out_